# EMF Induction Lab – data analysis

This notebook analyses the CSV files downloaded from the **EMF Lab** web page.

| File (from the web page) | Used in |
|---|---|
| `emf_raw_….csv` – *Record data → Download raw samples* | Part A: waveforms, frequency, amplitude, phase |
| `emf_faraday_….csv` – *Faraday's law → Download table* | Part B: ε₀ versus ω, slope = N·B·A |

**How to use:** put the CSV files in the same folder as this notebook (in Google Colab: click the folder icon on the left and upload them), type the file names in the first code cell, then choose *Run all*.

Only `numpy`, `pandas` and `matplotlib` are needed.

The file names below point to `sample_data/…_DEMO.csv`, which came from the web page's **Demo mode** (a simulated kit), so the notebook runs as-is. Replace them with your own files.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAW_FILE = "sample_data/emf_raw_DEMO.csv"          # your file, e.g. "emf_raw_2026-09-25_101500.csv"
FARADAY_FILE = "sample_data/emf_faraday_DEMO.csv"  # your file, e.g. "emf_faraday_2026-09-25_102300.csv"

C1_COLOR, C2_COLOR = "#b77f00", "#0a8ea6"   # same colours as the web page
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": 0.3})

## Part A – Raw waveforms

Each row of the raw file is one ADC reading: the time in seconds, which coil (1 or 2) and the EMF in millivolts.
The two coils are read alternately, so their time stamps are interleaved – always use each coil's own time column.

In [ ]:
raw = pd.read_csv(RAW_FILE)
c1 = raw[raw.coil == 1]
c2 = raw[raw.coil == 2]
t1, v1 = c1.time_s.to_numpy(), c1.emf_mV.to_numpy()
t2, v2 = c2.time_s.to_numpy(), c2.emf_mV.to_numpy()

duration = raw.time_s.iloc[-1] - raw.time_s.iloc[0]
print(f"{len(raw)} samples over {duration:.2f} s")
print(f"sample rate per coil: C1 {len(t1)/duration:.0f} /s, C2 {len(t2)/duration:.0f} /s")

show = t1 < t1[0] + 1.5          # first 1.5 s
plt.plot(t1[show], v1[show], color=C1_COLOR, label="Coil 1")
show = t2 < t2[0] + 1.5
plt.plot(t2[show], v2[show], color=C2_COLOR, label="Coil 2")
plt.xlabel("time (s)"); plt.ylabel("EMF (mV)"); plt.title("Induced EMF – first 1.5 s"); plt.legend()
plt.show()

### Frequency, amplitude and phase

1. **Frequency** – count the rising zero crossings of Coil 1 and divide by the time they span.
2. **Amplitude and phase** – at that frequency, fit each coil to
   $\varepsilon(t) = a\sin(2\pi f t) + b\cos(2\pi f t) + c$ by least squares.
   Then amplitude $= \sqrt{a^2+b^2}$ and phase $= \operatorname{atan2}(b, a)$.

The fit uses every sample, so it is less sensitive to noise than reading the peak off a graph.

In [ ]:
def rising_crossings(t, v):
    level = (v.max() + v.min()) / 2
    hyst = 0.1 * (v.max() - v.min())
    times, armed = [], False
    for i in range(1, len(v)):
        if v[i - 1] < level - hyst:
            armed = True
        if armed and v[i - 1] < level <= v[i]:
            times.append(t[i - 1] + (level - v[i - 1]) * (t[i] - t[i - 1]) / (v[i] - v[i - 1]))
            armed = False
    return np.array(times)

def sine_fit(t, v, f):
    w = 2 * np.pi * f
    M = np.column_stack([np.sin(w * t), np.cos(w * t), np.ones_like(t)])
    (a, b, c), *_ = np.linalg.lstsq(M, v, rcond=None)
    return np.hypot(a, b), np.degrees(np.arctan2(b, a)), c

x = rising_crossings(t1, v1)
f = (len(x) - 1) / (x[-1] - x[0])
A1, phi1, off1 = sine_fit(t1, v1, f)
A2, phi2, off2 = sine_fit(t2, v2, f)
lag = (phi1 - phi2 + 180) % 360 - 180      # positive: coil 2 lags coil 1

print(f"EMF frequency f = {f:.3f} Hz   (period {1000/f:.1f} ms)")
print(f"Coil 1: amplitude {A1:.1f} mV, DC offset {off1:+.2f} mV")
print(f"Coil 2: amplitude {A2:.1f} mV, DC offset {off2:+.2f} mV")
print(f"Amplitude ratio C2/C1 = {A2/A1:.3f}")
print(f"Coil 2 lags coil 1 by {lag:.1f} degrees")

In [ ]:
# Check the fit: data points and fitted sine for two cycles
w = 2 * np.pi * f
for t, v, A, phi, off, col, name in [(t1, v1, A1, phi1, off1, C1_COLOR, "Coil 1"),
                                      (t2, v2, A2, phi2, off2, C2_COLOR, "Coil 2")]:
    sel = t < t[0] + 2 / f
    tt = np.linspace(t[sel][0], t[sel][-1], 400)
    plt.plot(t[sel], v[sel], ".", ms=3, color=col, alpha=0.6, label=f"{name} data")
    plt.plot(tt, A * np.sin(w * tt + np.radians(phi)) + off, color=col, label=f"{name} fit")
plt.xlabel("time (s)"); plt.ylabel("EMF (mV)"); plt.title("Sine fit (two cycles)"); plt.legend(ncol=2)
plt.show()

## Part B – Faraday's law: ε₀ versus ω

For a magnet rotating at angular speed ω near a coil of N turns and area A, the peak EMF is
$\varepsilon_0 = N B A\,\omega$. A graph of ε₀ against ω should be a **straight line through the origin**
with slope $k = NBA$. Because 1 mV·s = 1 mWb, the slope in mV·s/rad is the flux linkage in milliweber-turns.

In [ ]:
far = pd.read_csv(FARADAY_FILE)
if far.clipped.any():
    print("Warning: rows", list(far.point[far.clipped == 1]), "were clipping at ±512 mV and are left out.")
far = far[far.clipped == 0]
print(far.to_string(index=False))

def fit_through_origin(x, y):
    k = np.sum(x * y) / np.sum(x * x)
    resid = y - k * x
    se = np.sqrt(np.sum(resid**2) / (len(x) - 1) / np.sum(x * x))
    r2 = 1 - np.sum(resid**2) / np.sum((y - y.mean())**2)
    return k, se, r2

w_ = far.omega_rad_s.to_numpy()
results = {}
for col, name, color in [("amp_C1_mV", "Coil 1", C1_COLOR), ("amp_C2_mV", "Coil 2", C2_COLOR)]:
    y = far[col].to_numpy()
    k, se, r2 = fit_through_origin(w_, y)
    (slope, intercept), cov = np.polyfit(w_, y, 1, cov=True)
    results[name] = k
    print(f"{name}: k = N·B·A = {k:.3f} ± {se:.3f} mV·s/rad (mWb-turns),  R² = {r2:.4f}")
    print(f"        free straight-line fit: intercept = {intercept:.2f} ± {np.sqrt(cov[1,1]):.2f} mV")

print(f"\nSlope ratio k2/k1 = {results['Coil 2']/results['Coil 1']:.3f}")

In [ ]:
fig, (ax, axr) = plt.subplots(2, 1, figsize=(8, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
wmax = w_.max() * 1.1
for col, name, color in [("amp_C1_mV", "Coil 1", C1_COLOR), ("amp_C2_mV", "Coil 2", C2_COLOR)]:
    y = far[col].to_numpy(); k = results[name]
    ax.plot(w_, y, "o", color=color, label=f"{name} data")
    ax.plot([0, wmax], [0, k * wmax], "--", color=color, label=f"{name}: ε₀ = {k:.2f}·ω")
    axr.plot(w_, y - k * w_, "o", color=color)
ax.set_xlim(0, wmax); ax.set_ylim(0, None)
ax.set_ylabel("EMF amplitude ε₀ (mV)"); ax.set_title("Faraday's law"); ax.legend()
axr.axhline(0, color="gray", lw=1)
axr.set_xlabel("angular speed ω (rad/s)"); axr.set_ylabel("residual (mV)")
plt.tight_layout(); plt.show()

### Estimate the magnetic field

Measure your coil: number of turns **N** and the area **A** of one turn (in m²). Then
$B = k / (N A)$ with k in V·s (divide mV·s by 1000). This is the *effective* field through the coil –
the flux is not uniform over the coil, so treat it as an average.

In [ ]:
N_TURNS = None        # e.g. 500
AREA_M2 = None        # e.g. 3.1e-4  (a 2 cm diameter coil: pi * 0.01**2)

if N_TURNS and AREA_M2:
    for name, k in results.items():
        B = (k / 1000) / (N_TURNS * AREA_M2)
        print(f"{name}: effective B = {B*1000:.1f} mT")
else:
    print("Enter N_TURNS and AREA_M2 above, then run this cell again.")

## Questions to discuss

1. Is the graph of ε₀ against ω a straight line? Does it pass through the origin? What could cause a small intercept?
2. Compare the slope ratio k₂/k₁ with the amplitude ratio from Part A. Should they agree? Why?
3. Coil 2 lags Coil 1 by some angle. Relate this angle to where the two coils sit around the magnet.
4. Why must readings above about ±500 mV be discarded? (Hint: look at the ADC range on the Help page.)